# Large Language Models (BERT / GPT)

_Modern Deep Learning & AI_

**Predict the next word over and over. Do it on the whole internet and language understanding falls out.**

A large language model (LLM) is a giant Transformer trained on a simple game: guess the next word.

     Two famous styles. GPT reads left to right and predicts the next token. BERT sees the whole sentence and predicts masked (hidden) tokens.

---

This notebook builds a GPT-style decoder block from scratch, piece by piece, and runs one next-token training step. Run each cell, read the note above it, and see how masked self-attention turns a sequence into next-word predictions. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

We'll build a single GPT-style decoder block and run one next-token training step. We do it in three parts: (1) the decoder block with masked self-attention, (2) the embedding, block, and output head wired together, (3) the next-token cross-entropy loss.

### Step 1 — Define a masked decoder block

A GPT decoder block has two sub-layers, each wrapped in a residual connection: **causal self-attention** followed by a **feed-forward** network. The triangular `mask` is what makes it causal — it blocks each position from attending to future tokens, so the model can only look leftward. LayerNorm is applied to the inputs of each sub-layer (pre-norm), and the `x + ...` residuals let gradients flow cleanly through deep stacks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DecoderBlock(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, heads, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d, 4 * d),
            nn.GELU(),
            nn.Linear(4 * d, d),
        )
        self.n1 = nn.LayerNorm(d)
        self.n2 = nn.LayerNorm(d)

    def forward(self, x):
        T = x.size(1)
        mask = torch.triu(torch.ones(T, T), diagonal=1).bool()  # block future tokens
        normed = self.n1(x)
        a, _ = self.attn(normed, normed, normed, attn_mask=mask)
        x = x + a                                  # residual after attention
        return x + self.ff(self.n2(x))             # residual after feed-forward

### Step 2 — Wire up embedding, block, and output head

An `Embedding` table turns each integer token id into a `d`-dimensional vector. We feed a tiny random sequence of 6 tokens through the embedding and the decoder block, then a final linear `head` projects each position back to a score (logit) for every word in the vocabulary. The result has shape `(1, T, vocab)`: one distribution-to-be per position.

In [ ]:
torch.manual_seed(0)
vocab = 50
d = 32
T = 6

emb = nn.Embedding(vocab, d)
block = DecoderBlock(d, 4)
head = nn.Linear(d, vocab)

tokens = torch.randint(0, vocab, (1, T))   # tiny synthetic sequence
hidden = block(emb(tokens))
logits = head(hidden)                      # (1, T, vocab)

### Step 3 — Compute the next-token loss

GPT is trained to predict each token from the ones before it. So we line up the logits at positions `0..T-2` against the *actual* tokens at positions `1..T-1` and take the cross-entropy. A lower loss means the model assigns higher probability to the true next token.

In [ ]:
# Next-token loss: predict token t+1 from positions up to t.
pred_logits = logits[:, :-1].reshape(-1, vocab)   # predictions at positions 0..T-2
targets = tokens[:, 1:].reshape(-1)               # the actual next tokens
loss = F.cross_entropy(pred_logits, targets)

print("logits shape:", logits.shape, "loss:", round(loss.item(), 3))

## Visualize the data & results

_After the real prompt The capital of France is, what is the next token at low vs high temperature?_

### Step 1 — Set up the next-token logits and a temperature softmax

After a real model reads 'The capital of France is', it produces a raw score (logit) for every candidate next word. Here are six plausible continuations with hand-picked logits. The `softmax_T` helper turns logits into probabilities, but first divides them by a **temperature** `T`: lower `T` sharpens the distribution toward the top choice, higher `T` flattens it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Next-token distribution for the prompt "The capital of France is".
labels = ["Paris", "a", "located", "the", "home", "now"]
logits = np.array([9.1, 4.2, 3.8, 3.1, 2.4, 1.7])

def softmax_T(z, T):
    s = z / T              # scale logits by temperature
    s -= s.max()           # subtract max for numerical stability
    e = np.exp(s)
    return e / e.sum()

### Step 2 — Compare a sharp vs a flat distribution

We run the same logits through the softmax at two temperatures: `T = 0.7` (sharp — almost all probability lands on 'Paris') and `T = 1.5` (flatter — the other words get a real chance). The bars show why temperature controls how 'creative' or 'random' a model's sampling feels.

In [ ]:
p_lo = softmax_T(logits, 0.7) * 100     # sharp distribution
p_hi = softmax_T(logits, 1.5) * 100     # flatter distribution

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
c = ["#7ee787"] + ["#4ea1ff"] * 5       # highlight the top token

axes[0].bar(labels, p_lo, color=c)
axes[0].set_title("next token at T = 0.7")
axes[0].set_ylabel("percent")

axes[1].bar(labels, p_hi, color="#ffb454")
axes[1].set_title("same logits at T = 1.5")

plt.show()